# Human Variation-Aware Alignment — Python Walkthrough

This notebook demonstrates the gen side of the variation-aware alignment workflow
using small synthetic data so it runs without downloading the human reference genome.

The full workflow (hg38 + GIAB data + vg alignment) is documented in `Analysis.md`.
The gen steps are identical; only the scale of the input data differs.

In [ ]:
import os
import tempfile

import gen

tmpdir = tempfile.mkdtemp()
repo = gen.Repository(os.path.join(tmpdir, "gen"))

## Import a reference sequence

In the full workflow this is a chromosome from hg38 (hundreds of MB).
Here we use a short synthetic sequence to keep the notebook self-contained.

In [ ]:
reference = (
    "ACGTACGTACGTACGTACGT"
    "TTTTGGGGCCCCAAAATTTT"
    "GCGCGCGCATATATATATAT"
    "AAACCCGGGTTTAGCTAGCT"
)

fasta_path = os.path.join(tmpdir, "chr1_region.fa")
with open(fasta_path, "w") as f:
    f.write(f">chr1\n{reference}\n")

result = repo.import_reference_fasta(fasta_path, "hg38")
print(result)

## Encode known variants into the graph

In the full workflow we apply the GIAB benchmark VCF (~120 MB compressed).
Here we encode two representative variants: a SNP and a small deletion.

In [ ]:
vcf_path = os.path.join(tmpdir, "known_variants.vcf")
with open(vcf_path, "w") as f:
    f.write(
        "##fileformat=VCFv4.1\n"
        "##contig=<ID=chr1,length=80>\n"
        "##FORMAT=<ID=GT,Number=1,Type=String,Description=\"Genotype\">\n"
        "#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\tFORMAT\tNA12878\n"
        "chr1\t5\t.\tC\tT\t50\tPASS\t.\tGT\t0/1\n"
        "chr1\t25\t.\tGGG\tG\t50\tPASS\t.\tGT\t1/1\n"
    )

result = repo.update_with_vcf(
    vcf_path,
    sample="NA12878",
    reference="hg38",
)
print(result)

## Inspect the variant graph

In [ ]:
sgs = repo.get_sequence_graphs()
print(f"{len(sgs)} sequence graph(s):")
for sg in sgs:
    print(f"  name={sg.name!r}  sample={sg.sample_name!r}")

## Export to GFA for graph alignment

In the full workflow the GFA is fed into `vg` for indexing and read alignment.
The GFA encodes the full variant graph including all alternate paths.

```sh
# Full workflow continues with:
# vg mod -X 32 graph.gfa > graph.mod.gfa
# vg autoindex -p index -w map -g graph.mod.gfa
# vg map -x index.xg -g index.gcsa -f R1.fq.gz -f R2.fq.gz > align.gam
# vg pack -e -x index.xg -g align.sorted.gam -o aln.pack
# vg call index.xg -k aln.pack > variants.vcf
```

In [ ]:
gfa_path = os.path.join(tmpdir, "graph.gfa")
repo.export_gfa(gfa_path)

with open(gfa_path) as f:
    lines = f.readlines()

segments = sum(1 for l in lines if l.startswith("S"))
links = sum(1 for l in lines if l.startswith("L"))
print(f"GFA: {segments} segments, {links} links")

## Search for sequence motifs across samples

In [ ]:
query = "ACGTACGT"
results = repo.search(query)

print(f"Matches for {query!r}:")
for sg, loci in results:
    if loci:
        print(f"  sample={sg.sample_name!r}: {len(loci)} match(es)")